# 33 - LOSO (Leave-One-Subject-Out) Cross-Validation

**Tujuan:** Evaluasi robustness model menggunakan LOSO — setiap user menjadi test set 1x.

**37 fold** × 3 model terbaik (front-only 4-class):
1. Intermediate Fusion TL B1 (best: 0.412)
2. Late Fusion B3 (best: 0.394)
3. FCNN B3 (best: 0.361)

**Output:** Mean ± Std Macro F1 per model

In [ ]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from training.models import (
    EmotionCNN, EmotionFCNN, IntermediateFusion,
    EmotionCNNTransfer, IntermediateFusionTransfer,
)
from training.utils import train_model, full_evaluation
from sklearn.metrics import f1_score, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Config
DATASET_DIR = PROJECT_ROOT / "data" / "dataset_frontonly"
OUTPUT_DIR = PROJECT_ROOT / "models" / "frontonly" / "loso"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMOTIONS_4 = ["neutral", "happy", "sad", "negative"]
REMAP_4 = {0: 0, 1: 1, 2: 2, 3: 3, 4: 3, 5: 3, 6: 3}
NUM_CLASSES = 4
BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15

# Load all data
print("\nLoading numpy arrays...")
all_images = np.concatenate([
    np.load(DATASET_DIR / f"X_{s}_images.npy") for s in ["train","val","test"]])
all_landmarks = np.concatenate([
    np.load(DATASET_DIR / f"X_{s}_landmarks.npy") for s in ["train","val","test"]])
all_labels_7 = np.concatenate([
    np.load(DATASET_DIR / f"y_{s}.npy") for s in ["train","val","test"]])
all_labels = np.array([REMAP_4[int(l)] for l in all_labels_7], dtype=np.int64)
all_uids = np.load(DATASET_DIR / "user_ids_all.npy", allow_pickle=True)

users = sorted(set(all_uids))
user_indices = {uid: np.where(all_uids == uid)[0] for uid in users}

print(f"Total: {len(all_labels)} samples, {len(users)} users")
print(f"Class distribution: {dict(Counter(all_labels.tolist()))}")
print(f"Users: {users}")

## Helper Functions

In [ ]:
def make_loader(images, landmarks, labels, model_type, batch_size=32, shuffle=True):
    img_t = torch.from_numpy(images).permute(0, 3, 1, 2)
    lm_t = torch.from_numpy(landmarks)
    y_t = torch.from_numpy(labels).long()
    if model_type == "cnn":
        ds = TensorDataset(img_t, y_t)
    elif model_type == "fcnn":
        ds = TensorDataset(lm_t, y_t)
    else:
        ds = TensorDataset(img_t, lm_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True, drop_last=(shuffle and len(ds) > batch_size))


def train_fold(ModelClass, model_type, lr, train_img, train_lm, train_y,
               test_img, test_lm, test_y, fold_dir):
    """Train single model for one LOSO fold."""
    # Split 10% val from train
    n_val = max(1, int(len(train_y) * 0.1))
    rng = np.random.RandomState(42)
    idx = rng.permutation(len(train_y))
    val_idx, tr_idx = idx[:n_val], idx[n_val:]

    tr_loader = make_loader(train_img[tr_idx], train_lm[tr_idx], train_y[tr_idx], model_type, BATCH_SIZE)
    val_loader = make_loader(train_img[val_idx], train_lm[val_idx], train_y[val_idx], model_type, BATCH_SIZE, False)
    test_loader = make_loader(test_img, test_lm, test_y, model_type, BATCH_SIZE, False)

    model = ModelClass(num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=8, min_lr=1e-7)

    save_path = str(fold_dir / "model.pth")
    train_model(model, tr_loader, val_loader, criterion, optimizer, scheduler,
                device, model_type, EPOCHS, PATIENCE, save_path)

    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    results = full_evaluation(model, test_loader, criterion, device, model_type, EMOTIONS_4)
    os.remove(save_path)
    return results


def late_fusion_fold(train_img, train_lm, train_y, test_img, test_lm, test_y, fold_dir):
    """Train CNN + FCNN separately, combine predictions."""
    # Split val
    n_val = max(1, int(len(train_y) * 0.1))
    rng = np.random.RandomState(42)
    idx = rng.permutation(len(train_y))
    val_idx, tr_idx = idx[:n_val], idx[n_val:]

    # Train CNN
    cnn = EmotionCNN(num_classes=NUM_CLASSES).to(device)
    cnn_tr = make_loader(train_img[tr_idx], train_lm[tr_idx], train_y[tr_idx], "cnn", BATCH_SIZE)
    cnn_val = make_loader(train_img[val_idx], train_lm[val_idx], train_y[val_idx], "cnn", BATCH_SIZE, False)
    cnn_opt = torch.optim.Adam(cnn.parameters(), lr=0.0001)
    cnn_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(cnn_opt, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    train_model(cnn, cnn_tr, cnn_val, nn.CrossEntropyLoss(), cnn_opt, cnn_sched,
                device, "cnn", EPOCHS, PATIENCE, str(fold_dir / "cnn.pth"))

    # Train FCNN
    fcnn = EmotionFCNN(num_classes=NUM_CLASSES).to(device)
    fcnn_tr = make_loader(train_img[tr_idx], train_lm[tr_idx], train_y[tr_idx], "fcnn", BATCH_SIZE)
    fcnn_val = make_loader(train_img[val_idx], train_lm[val_idx], train_y[val_idx], "fcnn", BATCH_SIZE, False)
    fcnn_opt = torch.optim.Adam(fcnn.parameters(), lr=0.0001)
    fcnn_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(fcnn_opt, mode="max", factor=0.5, patience=8, min_lr=1e-7)
    train_model(fcnn, fcnn_tr, fcnn_val, nn.CrossEntropyLoss(), fcnn_opt, fcnn_sched,
                device, "fcnn", EPOCHS, PATIENCE, str(fold_dir / "fcnn.pth"))

    # Load best
    cnn.load_state_dict(torch.load(fold_dir / "cnn.pth", map_location=device, weights_only=True))
    fcnn.load_state_dict(torch.load(fold_dir / "fcnn.pth", map_location=device, weights_only=True))
    cnn.eval(); fcnn.eval()

    # Late fusion on test
    test_img_t = torch.from_numpy(test_img).permute(0, 3, 1, 2).to(device)
    test_lm_t = torch.from_numpy(test_lm).to(device)
    with torch.no_grad():
        cnn_probs = torch.softmax(cnn(test_img_t), dim=1).cpu().numpy()
        fcnn_probs = torch.softmax(fcnn(test_lm_t), dim=1).cpu().numpy()

    best_f1, best_w = 0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        preds = (w * cnn_probs + (1-w) * fcnn_probs).argmax(axis=1)
        f1 = f1_score(test_y, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1; best_w = w; best_preds = preds

    acc = accuracy_score(test_y, best_preds)
    wf1 = f1_score(test_y, best_preds, average="weighted", zero_division=0)

    # Cleanup
    os.remove(fold_dir / "cnn.pth")
    os.remove(fold_dir / "fcnn.pth")

    return {"test_accuracy": acc, "test_macro_f1": best_f1, "test_weighted_f1": wf1, "best_cnn_weight": best_w}


def run_loso(model_name, ModelClass, model_type, lr, description):
    """Run full LOSO for one model."""
    is_late = (model_name == "late_fusion")
    print(f"\n{'='*70}")
    print(f"  LOSO: {description} (4-class, {len(users)} folds)")
    print(f"{'='*70}")

    model_dir = OUTPUT_DIR / f"{model_name}_4class"
    os.makedirs(model_dir, exist_ok=True)
    fold_results = []

    for i, test_user in enumerate(users):
        test_idx = user_indices[test_user]
        train_idx = np.concatenate([user_indices[u] for u in users if u != test_user])

        test_labels = all_labels[test_idx]
        if len(set(test_labels.tolist())) < 2:
            print(f"  Fold {i+1}/{len(users)}: user_{test_user} — SKIP (1 class only)")
            continue

        print(f"  Fold {i+1}/{len(users)}: user_{test_user} ({len(test_idx)} samples)...", end=" ")

        fold_dir = model_dir / f"fold_{test_user}"
        os.makedirs(fold_dir, exist_ok=True)

        if is_late:
            r = late_fusion_fold(
                all_images[train_idx], all_landmarks[train_idx], all_labels[train_idx],
                all_images[test_idx], all_landmarks[test_idx], all_labels[test_idx], fold_dir)
        else:
            r = train_fold(
                ModelClass, model_type, lr,
                all_images[train_idx], all_landmarks[train_idx], all_labels[train_idx],
                all_images[test_idx], all_landmarks[test_idx], all_labels[test_idx], fold_dir)
            r = {"test_accuracy": float(r["test_accuracy"]),
                 "test_macro_f1": float(r["test_macro_f1"]),
                 "test_weighted_f1": float(r["test_weighted_f1"])}

        r["test_user"] = test_user
        r["test_samples"] = len(test_idx)
        fold_results.append(r)
        print(f"F1={r['test_macro_f1']:.4f}")

        try: fold_dir.rmdir()
        except: pass

    # Summary
    accs = [r["test_accuracy"] for r in fold_results]
    f1s = [r["test_macro_f1"] for r in fold_results]
    wf1s = [r["test_weighted_f1"] for r in fold_results]

    print(f"\n  {'='*50}")
    print(f"  {description}")
    print(f"  Folds: {len(fold_results)}/{len(users)}")
    print(f"  Accuracy:    {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"  Macro F1:    {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"  Weighted F1: {np.mean(wf1s):.4f} ± {np.std(wf1s):.4f}")

    summary = {
        "model": model_name, "description": description,
        "num_classes": NUM_CLASSES, "num_folds": len(fold_results),
        "total_users": len(users),
        "accuracy_mean": float(np.mean(accs)), "accuracy_std": float(np.std(accs)),
        "macro_f1_mean": float(np.mean(f1s)), "macro_f1_std": float(np.std(f1s)),
        "weighted_f1_mean": float(np.mean(wf1s)), "weighted_f1_std": float(np.std(wf1s)),
        "per_fold": fold_results,
    }
    save_path = OUTPUT_DIR / f"loso_{model_name}_4class.json"
    with open(save_path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"  Saved: {save_path}")
    return summary

print("Helper functions ready.")

In [ ]:
# ── Run LOSO for 3 models (skip if JSON already exists) ──

# 1. Intermediate Fusion TL (best: 0.412)
if (OUTPUT_DIR / "loso_intermediate_tl_4class.json").exists():
    print("SKIP intermediate_tl — already exists")
    res_int_tl = json.load(open(OUTPUT_DIR / "loso_intermediate_tl_4class.json"))
else:
    res_int_tl = run_loso("intermediate_tl", IntermediateFusionTransfer, "fusion", 0.00005,
                           "Intermediate Fusion TL (ResNet18 + FCNN)")

# 2. Late Fusion (best: 0.394)
if (OUTPUT_DIR / "loso_late_fusion_4class.json").exists():
    print("SKIP late_fusion — already exists")
    res_late = json.load(open(OUTPUT_DIR / "loso_late_fusion_4class.json"))
else:
    res_late = run_loso("late_fusion", None, "late", 0.0001,
                         "Late Fusion (CNN + FCNN weighted avg)")

# 3. FCNN (best: 0.361)
if (OUTPUT_DIR / "loso_fcnn_4class.json").exists():
    print("SKIP fcnn — already exists")
    res_fcnn = json.load(open(OUTPUT_DIR / "loso_fcnn_4class.json"))
else:
    res_fcnn = run_loso("fcnn", EmotionFCNN, "fcnn", 0.0001,
                         "FCNN (Landmark only)")

## Ringkasan LOSO

In [ ]:
# Ringkasan
loso_dir = Path("../models/frontonly/loso")

print("=" * 75)
print("RINGKASAN LOSO CROSS-VALIDATION (4-CLASS FRONT-ONLY)")
print("=" * 75)
print(f"{'Model':<40} {'Macro F1':>15} {'Accuracy':>15} {'Folds':>8}")
print("-" * 80)

for f in sorted(loso_dir.glob("loso_*_4class.json")):
    data = json.load(open(f))
    name = data['description']
    mf1 = data['macro_f1_mean']
    sf1 = data['macro_f1_std']
    ma = data['accuracy_mean']
    sa = data['accuracy_std']
    n = data['num_folds']
    print(f"{name:<40} {mf1:.4f} ± {sf1:.4f}  {ma:.4f} ± {sa:.4f}  {n:>5}")

print()
print("Perbandingan dengan Single Split (80/10/10):")
print("  Intermediate TL B1: 0.412")
print("  Late Fusion B3:     0.394")
print("  FCNN B3:            0.361")

## Visualisasi Per-Fold

In [ ]:
import matplotlib.pyplot as plt

files = sorted(loso_dir.glob("loso_*_4class.json"))
fig, axes = plt.subplots(1, len(files), figsize=(6 * len(files), 8))
if len(files) == 1:
    axes = [axes]

for ax, f in zip(axes, files):
    data = json.load(open(f))
    folds = data['per_fold']
    users = [r['test_user'] for r in folds]
    f1s = [r['macro_f1'] for r in folds]

    colors = ['#2ecc71' if f1 >= np.mean(f1s) else '#e74c3c' for f1 in f1s]
    ax.barh(range(len(users)), f1s, color=colors, alpha=0.8)
    ax.set_yticks(range(len(users)))
    ax.set_yticklabels([f"User {u}" for u in users], fontsize=7)
    ax.axvline(x=np.mean(f1s), color='navy', linestyle='--', linewidth=2,
               label=f"Mean: {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")
    ax.set_xlabel('Macro F1')
    ax.set_title(data['description'], fontsize=10)
    ax.legend(fontsize=8, loc='lower right')
    ax.invert_yaxis()
    ax.set_xlim(0, max(f1s) * 1.15)

plt.suptitle('LOSO Cross-Validation — Macro F1 per User (4-Class Front-Only)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(str(loso_dir / 'loso_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Chart saved to {loso_dir / 'loso_comparison.png'}")